# pandas: Series and DataFrame

## Introduction

pandas is a library for data manipulation and analysis. Created in 2008 in response to the increasing use of Python in scientific applications traditionally dominated by **R**, MATLAB or SAS, and building on the maturity and stability of **NumPy** and **SciPy**. Its name derives from ***Pan**el **Da**ta*, a common term in statistics and econometrics for multidimensional datasets.

It allows:
- Easy importing from CSV, JSON, Excel, SQL, etc.
- Manipulation operations: selection, filtering, aggregation.
- Data cleaning (also called *data cleansing*).
- *Data wrangling* or *data munging*: transforming data between formats.

pandas has two structures:
- **Series**: a one-dimensional array with labels
- **DataFrame**: a two-dimensional table, with labelled rows and columns

There used to be a third one, the three-dimensional `Panel`, removed in pandas 0.25 (2019): a DataFrame with a hierarchical index (a *MultiIndex*) does the same job.

Official documentation: https://pandas.pydata.org/docs/

In [ ]:
import pandas as pd

## Series

The **Series** type is a one-dimensional array that contains a sequence of values and a sequence of labels associated with the values, called an index. The existence of this explicit index (which can be of any immutable type) is the main difference from a NumPy vector, which has an implicit index (a sequence of integers indicating the position). Series indexes are like dictionary indexes, while **NumPy** indexes are like list indexes.

### Structure

In [ ]:
example_series = pd.Series([1, 2, 3, 4, 5, 6])  # Series with implicit index since it starts from a list
print(example_series)
print(type(example_series))

0    1
1    2
2    3
3    4
4    5
5    6
dtype: int64
<class 'pandas.Series'>


Just as NumPy creates arrays from lists, pandas creates a Series from a dictionary: the keys become the index, and the dictionary's values the values of the Series.

In [ ]:
student_grades = pd.Series({'Student 1': 5, 'Student 2': 10, 'Student 3': 7, 'Student 4': 8})
student_grades

Student 1     5
Student 2    10
Student 3     7
Student 4     8
dtype: int64

In [ ]:
pd.Series([5, 10, 7, 8], index=["Student 1", "Student 2", "Student 3", "Student 4"])  # the same, from values and labels

Student 1     5
Student 2    10
Student 3     7
Student 4     8
dtype: int64

In [ ]:
occupied_seats = pd.Series({1: "Pepe Pérez", 7: "Juan Gómez", 6: "Ana López", 2: "María García", 5: "Luisa Martínez"})
occupied_seats

1        Pepe Pérez
7        Juan Gómez
6         Ana López
2      María García
5    Luisa Martínez
dtype: str

### Accessing elements of a series

Be careful when the labels of the index are numbers: a label and a position can then be the same number and point to different elements.

To select by label, use the **loc** attribute; to select by position, use **iloc** (from *integer location*). The indexing operator **[]** alone also selects by label, as with a dictionary.

In [ ]:
print(occupied_seats[7])  # the value with label 7
print(occupied_seats.loc[7])  # the same, explicitly by label
print(occupied_seats.iloc[1])  # the value at position 1

Juan Gómez
Juan Gómez
Juan Gómez


In [ ]:
print(student_grades["Student 1"])  # the value with label "Student 1"
print(student_grades.loc["Student 1"])  # the same, explicitly by label
print(student_grades.iloc[0])  # the value at position 0

5
5
5


An integer inside **[]** is always a label, even when the labels are text: `student_grades[0]` looks for a label `0`, which doesn't exist. Before pandas 3.0 it fell back to position 0, with a warning; now it raises a `KeyError`:

In [ ]:
try:
    student_grades[0]
except KeyError as error:
    print(f"KeyError: {error}")

KeyError: 0


An integer *slice* inside **[]**, however, still means positions, even when the labels are integers: `occupied_seats[0:2]` gives the first two seats, not the seats labelled 0 and 1. That inconsistency is why **loc** and **iloc** are the safe choice whenever a number is involved.

In [ ]:
occupied_seats[0:2]

1    Pepe Pérez
7    Juan Gómez
dtype: str

In [ ]:
# Modifying values by labels
student_grades['Student 1'] = 10
student_grades['Student 3':] = 5  # from label 'Student 3' to the end: a slice of labels includes both ends
student_grades

Student 1    10
Student 2    10
Student 3     5
Student 4     5
dtype: int64

In [ ]:
print(student_grades.mean())  # mean of the grades
print(student_grades.std())  # standard deviation

7.5
2.886751345948129


In [ ]:
print(student_grades.describe())  # descriptive statistics of the grades

count     4.000000
mean      7.500000
std       2.886751
min       5.000000
25%       5.000000
50%       7.500000
75%      10.000000
max      10.000000
dtype: float64


## DataFrame

### Structure of a DataFrame

A **DataFrame** is a two-dimensional tabular data structure, with labelled rows and columns. It is similar to a relational database table (SQL). It can be considered as a collection of Series that share the same index. It is the most used data structure in pandas.

In [ ]:
pd.DataFrame({'Grades': student_grades})  # a DataFrame from the Series, naming its column

,Grades
Student 1,10
Student 2,10
Student 3,5
Student 4,5


In [ ]:
# A DataFrame with the grades of several students in several modules
pd.DataFrame({'PIA': student_grades, 'SAA': [5, 6, 7, 8], 'MIA': [9, 8, 7, 6], 'SBD': [10, 9, 8, 7], 'BDA': [6, 7, 8, 9]})

,PIA,SAA,MIA,SBD,BDA
Student 1,10,5,9,10,6
Student 2,10,6,8,9,7
Student 3,5,7,7,8,8
Student 4,5,8,6,7,9


In the previous case, the index of `student_grades` became the index of the DataFrame. The first column comes from a Series, and the following ones from lists.

In [ ]:
# Another option is to give the index explicitly, with all the other grades as lists
pd.DataFrame({'PIA': student_grades, 'SAA': [5, 6, 7, 8], 'MIA': [9, 8, 7, 6], 'SBD': [10, 9, 8, 7], 'BDA': [6, 7, 8, 9]},
             index=['Wrong Name', 'Student 2', 'Student 3', 'Student 4'])

,PIA,SAA,MIA,SBD,BDA
Wrong Name,NaN,5,9,10,6
Student 2,10.0,6,8,9,7
Student 3,5.0,7,7,8,8
Student 4,5.0,8,6,7,9


We made an error in a student's name, and since the first grade list (PIA) was a Series, it doesn't find the grade for the index 'Wrong Name' and returns **NaN (Not a Number)** (a NumPy constant). To avoid this, we can create a DataFrame from a dictionary of lists instead of a dictionary of Series. The other grades are simple lists without an index, so they are assumed to be correct.
However, in this type of process it is important to be alert. Having each student's grades only identified by their position in a list is not very robust, since if a student is added or the order of students is changed, the grades will be assigned to different students. It is better to use a dictionary of Series, since the explicit index allows correctly identifying each student.

The following solution is more robust:

In [ ]:
grades_pia = pd.Series({'Marvin Minsky': 5.7, 'John McCarthy': 6.5, 'Claude Shannon': 6.5, 'Alan Turing': 7.0})
grades_saa = pd.Series({'Marvin Minsky': 8.0, 'John McCarthy': 8.5, 'Claude Shannon': 8.0, 'Alan Turing': 9.0})
grades_mia = pd.Series({'Marvin Minsky': 7.0, 'John McCarthy': 6.0, 'Claude Shannon': 6.0, 'Alan Turing': 7.0})
grades_sbd = pd.Series({'Marvin Minsky': 9.0, 'John McCarthy': 9.0, 'Claude Shannon': 9.0, 'Alan Turing': 10.0})
grades_bda = pd.Series({'John McCarthy': 7.8, 'Claude Shannon': 6.9, 'Alan Turing': 9.9, 'Marvin Minsky': 10})  # another order: Series are aligned by label

grades_df = pd.DataFrame({'PIA': grades_pia, 'SAA': grades_saa, 'MIA': grades_mia, 'SBD': grades_sbd, 'BDA': grades_bda})
grades_df

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


A couple of problems to consider when using *strings* as indexes in data analysis:
- They may not be unique (two people can have the same name)
- There may be variations on how names are written (for example, with uppercase or lowercase) in different data sources.
These are two of the reasons why relational databases always use unique indexed primary keys, often auto-incremental integers that have no meaning in themselves (surrogate keys).

In [ ]:
grades_pia = pd.Series({'Marvin Minsky': 5.7, 'John McCarthy': 6.2, 'Claude Shannon': 6.5, 'Alan Turing': 7.0})
grades_saa = pd.Series({'marvin minsky': 8.0, 'McCarthy': 8.5, 'shannon': 8.0, 'Alan-Turing': 9.0})
grades_df_messy = pd.DataFrame({'PIA': grades_pia, 'SAA': grades_saa})
grades_df_messy

,PIA,SAA
Alan Turing,7.0,NaN
Alan-Turing,NaN,9.0
Claude Shannon,6.5,NaN
John McCarthy,6.2,NaN
Marvin Minsky,5.7,NaN
McCarthy,NaN,8.5
marvin minsky,NaN,8.0
shannon,NaN,8.0


### Information about a DataFrame

In [ ]:
grades_df.info()  # information about the DataFrame

<class 'pandas.DataFrame'>
Index: 4 entries, Alan Turing to Marvin Minsky
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PIA     4 non-null      float64
 1   SAA     4 non-null      float64
 2   MIA     4 non-null      float64
 3   SBD     4 non-null      float64
 4   BDA     4 non-null      float64
dtypes: float64(5)
memory usage: 244.0 bytes


In [ ]:
grades_df.head()  # First 5 rows (in this case, there are only 4, normally we will work with much larger datasets and it will be useful to see only the first rows to get an idea of the data)

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


In [ ]:
grades_df.shape  # number of rows and columns

(4, 5)

In [ ]:
grades_df.keys()  # "Index" object with column names and type

Index(['PIA', 'SAA', 'MIA', 'SBD', 'BDA'], dtype='str')

In [ ]:
grades_df.columns  # Equivalent to the above but only for DataFrame (the keys() method also works to retrieve Series keys)

Index(['PIA', 'SAA', 'MIA', 'SBD', 'BDA'], dtype='str')

In [ ]:
grades_df.dtypes  # data types of columns

PIA    float64
SAA    float64
MIA    float64
SBD    float64
BDA    float64
dtype: object

In [ ]:
grades_df.index  # row labels

Index(['Alan Turing', 'Claude Shannon', 'John McCarthy', 'Marvin Minsky'], dtype='str')

### Writing and reading data files

Pandas offers a wide variety of functions to import and export data from and to files. Without going into depth, as an example we can store the DataFrame `grades_df` in a **CSV file** with the **to_csv** function and retrieve it with the **read_csv** function.

In [ ]:
grades_df.to_csv('data/grades.csv')  # the data directory must exist
df = pd.read_csv('data/grades.csv', index_col=0)
df

,PIA,SAA,MIA,SBD,BDA
Alan Turing,7.0,9.0,7.0,10.0,9.9
Claude Shannon,6.5,8.0,6.0,9.0,6.9
John McCarthy,6.5,8.5,6.0,9.0,7.8
Marvin Minsky,5.7,8.0,7.0,9.0,10.0


The parameter `index_col=0` indicates that the first column of the csv file is the explicit index of the DataFrame, if not specified an implicit index is created.

In [ ]:
pd.read_csv('data/grades.csv')

,Unnamed: 0,PIA,SAA,MIA,SBD,BDA
0,Alan Turing,7.0,9.0,7.0,10.0,9.9
1,Claude Shannon,6.5,8.0,6.0,9.0,6.9
2,John McCarthy,6.5,8.5,6.0,9.0,7.8
3,Marvin Minsky,5.7,8.0,7.0,9.0,10.0
